In [1]:
from dotenv import load_dotenv
import os
load_dotenv()


True

elijo la base

In [2]:
from pymongo import MongoClient
import os

uri = os.getenv("URI_mia")
client = MongoClient(uri)

db = client["catalogo_repuestos"]        
coleccion = db["repuestos_internos"]

elegir modelos para embeddings

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/home/nicolas/entornos/trabajo-final-modulo6_python3.11.13/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


función para búsquedas semánticas

In [ ]:
def buscar_semantico(texto, n=5):
    # 1. Vectorizamos la consulta
    query_vector = embedder.embed_query(texto)

    # 2. Ejecutamos el vectorSearch
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",      
                "path": "embedding",
                "queryVector": query_vector,
                "numCandidates": 100,
                "limit": n
            }
        }
    ]

    resultados = coleccion.aggregate(pipeline)
    return list(resultados)


las búsquedas semánticas

In [11]:
res = buscar_semantico("pastillas de freno", n=5)

for r in res:
    print(r["Descripción"], "-", r["Marca"], "- (", r["Marca Vehículo"], r["Modelo"], r["Año"],") - $", r.get("Precio"))
    
#print(res)

Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694
Pastillas de freno - Ferodo - ( Ford Focus 2006-2012 ) - $ 110840
Pastillas de freno - Brembo - ( Chevrolet Cruze 2006-2012 ) - $ 115248
Pastillas de freno - Brembo - ( Fiat Uno 2020-2024 ) - $ 190180
Pastillas de freno - Ferodo - ( Toyota Hilux 2000-2005 ) - $ 56204


In [16]:
res = buscar_semantico("Marca: Toyota; Modelo: Etios; año: 2009; Descripción: pastillas de freno", n=5)
for r in res:
    print(r["Descripción"], "-", r["Marca"], "- (", r["Marca Vehículo"], r["Modelo"], r["Año"],") - $", r.get("Precio"))
    
#print(res)

Pastillas de freno - TRW - ( Toyota Corolla 2006-2012 ) - $ 57096
Pastillas de freno - Ferodo - ( Toyota Hilux 2000-2005 ) - $ 56204
Amortiguador delantero - Sachs - ( Toyota Corolla 2000-2005 ) - $ 91018
Pastillas de freno - Ferodo - ( Chevrolet Corsa 2020-2024 ) - $ 199914
Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694
